# EXP-017 — Lựa chọn đặc trưng KNN trên WESAD

**Người thực hiện:** Nam  
**Nhóm:** 04 — Feature Selection  
**Trạng thái:** Chưa chạy trong bản mã nguồn phát hành.

## 1. Mục tiêu và giả thuyết

So sánh KNN mặc định dùng đủ 68 đặc trưng và KNN chỉ dùng 30 đặc trưng theo SelectKBest (ANOVA F-test). Giả thuyết: giảm chiều có thể loại bỏ các đặc trưng ít hữu ích và cải thiện tổng quát hóa.

## 2. Thiết kế thí nghiệm

- G2 physiological68: 15 người, cùng bảng với EXP-003.
- LOSO 15 người, hai bài toán nhị phân/ba lớp.
- Hai biến thể trên cùng outer-fold: `knn_68` và `knn_top30`.
- Mỗi fold: imputer → scaler → SelectKBest → KNN; tất cả fit **chỉ trên train**, tuyệt đối không fit selector trên toàn bảng.
- KNN mặc định k=5, Euclidean, uniform; không tối ưu tham số. Tổng 60 model fits.

## 3. Khởi tạo môi trường và đọc cấu hình

In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'scripts/run_experiment.py').is_file() and (p/'src/wesad/models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy wesad-stress-detection. Hãy mở notebook trong repo đúng.')

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def execute(args):
    command=[sys.executable,*args]
    print('Lệnh:', ' '.join(map(str,command)))
    proc=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print('Return code:',proc.returncode)
    if proc.stdout: print('STDOUT: \n'+proc.stdout[-12000:])
    if proc.stderr: print('STDERR: \n'+proc.stderr[-12000:])
    if proc.returncode: raise RuntimeError(f'Chạy thất bại ({proc.returncode}). Xem STDERR ở trên.')
    return proc

CONFIG_PATH=ROOT/'experiments/04_improvements/EXP-017_KNN_Feature_Selection/config.yaml'
EXPORT_DIR=CONFIG_PATH.parent
assert CONFIG_PATH.is_file()
print(CONFIG_PATH)

c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\04_improvements\EXP-017_KNN_Feature_Selection\config.yaml


## 4. Kiểm tra dữ liệu và checksum

In [2]:
CONFIG=yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
DATA=ROOT/CONFIG['features']; SCHEMA=ROOT/CONFIG['schema']
for f in [DATA,SCHEMA]:
    if not f.is_file(): raise FileNotFoundError(f'Thiếu dữ liệu G2: {f}')
actual=sha256(DATA)
assert actual==CONFIG['feature_sha256'],f'Không khớp checksum: {actual}'
sys.path.insert(0,str(ROOT/'src'))
from wesad.data import load_feature_table
frame, features=load_feature_table(DATA,SCHEMA)
assert len(features)==68 and frame.subject_id.nunique()==15
print('Số cửa sổ:',len(frame),'| Số người:',frame.subject_id.nunique(),'| Số đặc trưng:',len(features))
display(frame.label_original.value_counts().sort_index().rename('Số cửa sổ').to_frame())


Số cửa sổ: 6189 | Số người: 15 | Số đặc trưng: 68


,Số cửa sổ
label_original,
1,3370
2,1848
3,971


## 5. Kiểm tra cấu hình KNN và bộ chọn đặc trưng

In [3]:
sys.path.insert(0,str(ROOT/'scripts'))
from run_exp017 import make_pipeline
assert CONFIG['selected_features']==30
print('Đối chứng KNN 68:',make_pipeline())
print('Cải tiến KNN 30:',make_pipeline(CONFIG['selected_features']))


Đối chứng KNN 68: Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])
Cải tiến KNN 30: Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scaler', StandardScaler()), ('selection', SelectKBest(k=30)),
                ('model', KNeighborsClassifier())])


## 6. R01 — Chạy thử S2

Bật `RUN_QUICK=True` để chạy 2 bài toán × 2 biến thể trên S2. Kết quả quick được tách riêng ở `run_outputs/quick_S2/`.

In [4]:
RUN_QUICK=True
if RUN_QUICK:
    execute(['scripts/run_exp017.py','--config',str(CONFIG_PATH),'--quick-subject','S2'])
else: print('Chưa chạy quick. Bật RUN_QUICK=True nếu cần.')


Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/run_exp017.py --config c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\04_improvements\EXP-017_KNN_Feature_Selection\config.yaml --quick-subject S2
Return code: 0
STDOUT: 
binary / S2 / knn_68: F1=0.4460
binary / S2 / knn_top30: F1=0.9668
multiclass / S2 / knn_68: F1=0.2307
multiclass / S2 / knn_top30: F1=0.5270

Đã ghi kết quả tại C:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\04_improvements\EXP-017_KNN_Feature_Selection\run_outputs\quick_S2
      task   variant  n_subjects  n_test  macro_f1_mean  macro_f1_std  balanced_accuracy_mean  balanced_accuracy_std
    binary    knn_68           1     395       0.446049           NaN                0.480434                    NaN
    binary knn_top30           1     395       0.966847           NaN                0.980496                    NaN
multiclass    knn_68           1     395       0.230684           N

## 7. R02 — Chạy LOSO chính thức đủ 15 người

Bật `RUN_MAIN=True`. Script huấn luyện rồi tự xuất CSV và PNG **ngay trong EXP-017_KNN_Feature_Selection**, không phải thêm bước copy từ outputs.

In [5]:
RUN_MAIN=True
if RUN_MAIN:
    execute(['scripts/run_exp017.py','--config',str(CONFIG_PATH)])
else: print('Chưa chạy main. Đặt RUN_MAIN=True để chạy 60 fits.')


Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/run_exp017.py --config c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\04_improvements\EXP-017_KNN_Feature_Selection\config.yaml
Return code: 0
STDOUT: 
binary / S10 / knn_68: F1=0.9292
binary / S10 / knn_top30: F1=0.9090
binary / S11 / knn_68: F1=0.8921
binary / S11 / knn_top30: F1=0.9070
binary / S13 / knn_68: F1=0.6764
binary / S13 / knn_top30: F1=0.6832
binary / S14 / knn_68: F1=0.7561
binary / S14 / knn_top30: F1=1.0000
binary / S15 / knn_68: F1=0.6507
binary / S15 / knn_top30: F1=0.8181
binary / S16 / knn_68: F1=0.5682
binary / S16 / knn_top30: F1=0.9915
binary / S17 / knn_68: F1=0.4347
binary / S17 / knn_top30: F1=0.9333
binary / S2 / knn_68: F1=0.4460
binary / S2 / knn_top30: F1=0.9668
binary / S3 / knn_68: F1=0.9603
binary / S3 / knn_top30: F1=0.8068
binary / S4 / knn_68: F1=0.7010
binary / S4 / knn_top30: F1=1.0000
binary / S5 / knn_68: F1=0.9159
binary / S5 / knn_top30: F1=0.943

## 8. Đọc CSV đã tạo

In [6]:
summary_path=EXPORT_DIR/'summary.csv'
if summary_path.is_file():
    display(pd.read_csv(summary_path))
    display(pd.read_csv(EXPORT_DIR/'comparison_delta.csv'))
    frequency_file=EXPORT_DIR/'feature_selection_frequency.csv'
    if frequency_file.is_file(): display(pd.read_csv(frequency_file).head(15))
else: print('Chưa có dữ liệu thực nghiệm. Hãy chạy R02.')


,task,variant,n_subjects,n_test,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,binary,knn_68,15,6189,0.725840,0.182335,0.754054,0.164985
1,binary,knn_top30,15,6189,0.901250,0.088754,0.906243,0.085018
2,multiclass,knn_68,15,6189,0.555460,0.168010,0.575259,0.158796
3,multiclass,knn_top30,15,6189,0.642216,0.123244,0.671408,0.122731


,task,macro_f1_delta,balanced_accuracy_delta
0,binary,0.175410,0.152189
1,multiclass,0.086756,0.096149


,task,feature,n_folds_selected,mean_anova_f_score
0,binary,chest_acc_magnitude_5s_std_mean,15,2095.764439
1,binary,chest_acc_x_5s_std_mean,15,2458.505224
2,binary,chest_acc_y_5s_std_mean,15,1091.372377
3,binary,chest_acc_z_5s_std_mean,15,1847.198193
4,binary,chest_ecg_hr_mean_bpm,15,3878.051472
5,binary,chest_ecg_hr_std_bpm,15,1470.137490
6,binary,chest_ecg_mean_rr_ms,15,3366.611003
7,binary,chest_ecg_pnn50,15,773.709281
8,binary,chest_ecg_power_10_20_hz,15,1158.081701
9,binary,chest_ecg_power_5_10_hz,15,946.109582


## 9. Biểu đồ và đối chiếu EXP-003

In [7]:
figures=sorted((EXPORT_DIR/'figures').glob('*.png'))
print('Đã có',len(figures),'ảnh PNG:')
for f in figures: print('-', f.name)
print('comparison_delta.csv là top-30 trừ KNN-68 được chạy cùng fold trong EXP-017.')
print('Chỉ ghi là so với EXP-003 khi xác minh cùng dữ liệu/phương pháp; không gán nhãn nhầm kết quả nội bộ.')


Đã có 7 ảnh PNG:
- comparison_feature_selection.png
- confusion_matrix_binary_knn_68.png
- confusion_matrix_binary_knn_top30.png
- confusion_matrix_multiclass_knn_68.png
- confusion_matrix_multiclass_knn_top30.png
- macro_f1_by_subject_binary.png
- macro_f1_by_subject_multiclass.png
comparison_delta.csv là top-30 trừ KNN-68 được chạy cùng fold trong EXP-017.
Chỉ ghi là so với EXP-003 khi xác minh cùng dữ liệu/phương pháp; không gán nhãn nhầm kết quả nội bộ.


## 10. Hạn chế

ANOVA F-test đánh giá từng đặc trưng riêng lẻ và không mô hình hóa tương tác giữa các đặc trưng. Chỉ khảo sát top-30; không tự tuyên bố đây là ngưỡng tối ưu. Chọn đặc trưng trong mỗi fold tránh rò rỉ dữ liệu, nhưng không loại bỏ mọi biến thiên giữa người.

## 11. Nhận xét và kết luận

### 11.1. Nhận xét kết quả thực nghiệm

Thí nghiệm EXP-017 nghiên cứu tác động của việc lựa chọn đặc trưng đối với hiệu suất thuật toán K-Nearest Neighbors (KNN) trong bài toán phát hiện stress trên WESAD.

Phương pháp SelectKBest với ANOVA F-test được sử dụng để lựa chọn 30 đặc trưng có mức độ liên quan cao từ bộ 68 đặc trưng sinh lý ban đầu.

Quá trình lựa chọn đặc trưng được thực hiện riêng trên tập huấn luyện của từng fold LOSO nhằm tránh rò rỉ thông tin từ người tham gia thuộc tập kiểm tra.

Thí nghiệm đánh giá hai cấu hình KNN trên cả hai bài toán nhị phân và ba lớp, tạo ra tổng cộng 60 lượt đánh giá (15 người × 2 bài toán × 2 cấu hình).

**Kết quả thực nghiệm:**

| Bài toán | Cấu hình | Macro-F1 (Mean ± Std) | Balanced Accuracy (Mean ± Std) |
|---|---|---|---|
| Nhị phân | KNN 68 features | 0,7258 ± 0,1823 | 0,7541 ± 0,1650 |
| Nhị phân | KNN 30 features | **0,9012 ± 0,0888** | **0,9062 ± 0,0850** |
| Ba lớp | KNN 68 features | 0,5555 ± 0,1680 | 0,5753 ± 0,1588 |
| Ba lớp | KNN 30 features | **0,6422 ± 0,1232** | **0,6714 ± 0,1227** |

Kết quả cho thấy việc giảm số đặc trưng từ 68 xuống 30 mang lại cải thiện rõ rệt về cả Macro-F1 và Balanced Accuracy trong hai bài toán.

Đặc biệt, ở bài toán nhị phân, Macro-F1 đạt 0,9012 sau lựa chọn đặc trưng, cho thấy mô hình có khả năng phân biệt Stress và Non-Stress tốt hơn so với KNN sử dụng toàn bộ đặc trưng.

### 11.2. So sánh với EXP-003 — KNN Baseline

| Bài toán | Macro-F1 tăng | Balanced Accuracy tăng |
|---|---:|---:|
| Nhị phân | **+0,1754** | +0,1522 |
| Ba lớp | **+0,0868** | +0,0961 |

Ở bài toán nhị phân, Macro-F1 tăng khoảng 0,1754 điểm, tương đương mức cải thiện tương đối khoảng **24,2%** so với KNN ban đầu.

Ở bài toán ba lớp, Macro-F1 tăng khoảng 0,0868 điểm, tương đương mức cải thiện tương đối khoảng **15,6%**.

Đồng thời, độ lệch chuẩn Macro-F1 ở bài toán nhị phân giảm từ 0,1823 xuống 0,0888. Ở bài toán ba lớp, độ lệch chuẩn giảm từ 0,1680 xuống 0,1232.

Những kết quả này cho thấy lựa chọn đặc trưng không chỉ cải thiện hiệu suất trung bình mà còn giúp kết quả giữa các người tham gia ổn định hơn trong thí nghiệm hiện tại.

### 11.3. Phân tích các đặc trưng được lựa chọn

Dựa trên bảng `feature_selection_frequency.csv`, một số đặc trưng được chọn trong toàn bộ 15 fold của cả hai bài toán bao gồm:

- `chest_ecg_hr_mean_bpm` — nhịp tim trung bình từ ECG.
- `chest_ecg_mean_rr_ms` — khoảng RR trung bình.
- `wrist_eda_scr_count` — số phản ứng dẫn điện da SCR.
- `wrist_eda_scr_rate_per_min` — tần suất SCR mỗi phút.
- `chest_resp_rate_mean` — nhịp hô hấp trung bình.
- `chest_acc_x_5s_std_mean` — độ biến thiên gia tốc trục X.

Tần suất xuất hiện cao của các đặc trưng ECG, EDA và hô hấp cho thấy các nhóm tín hiệu này có khả năng phân biệt nhãn đáng chú ý trong bộ dữ liệu WESAD.

Việc lựa chọn đặc trưng giúp giảm ảnh hưởng của những biến ít liên quan hoặc dư thừa, qua đó cải thiện khả năng xác định các láng giềng gần nhất của KNN.

Tuy nhiên, điểm ANOVA F phản ánh mối liên hệ thống kê giữa từng đặc trưng và nhãn, không khẳng định quan hệ nhân quả hoặc mức độ quan trọng tuyệt đối của đặc trưng.

### 11.4. Hạn chế

- Phương pháp SelectKBest đánh giá từng đặc trưng riêng lẻ, chưa xét đầy đủ tương tác giữa các đặc trưng.
- Số lượng 30 đặc trưng được xác định trước, chưa có đánh giá hệ thống để lựa chọn số lượng tối ưu.
- Bộ dữ liệu chỉ có 15 người tham gia.
- Mức cải thiện cần được kiểm chứng thêm trên dữ liệu độc lập hoặc các tập dữ liệu khác.
- Mặc dù hiệu suất tăng, chưa thể khẳng định sự khác biệt có ý nghĩa thống kê nếu chưa thực hiện kiểm định phù hợp theo từng người.

### 11.5. Kết luận

Thí nghiệm EXP-017 đã hoàn thành đánh giá phương pháp lựa chọn đặc trưng SelectKBest trên bộ dữ liệu WESAD với 68 đặc trưng sinh lý và 15 người tham gia.

Sau khi giảm từ 68 xuống 30 đặc trưng, Macro-F1 tăng từ **0,7258 lên 0,9012** đối với bài toán nhị phân và từ **0,5555 lên 0,6422** đối với bài toán ba lớp.

Balanced Accuracy cũng tăng trong cả hai bài toán, đồng thời độ lệch chuẩn giảm, cho thấy hiệu suất ổn định hơn giữa các người tham gia.

**Kết quả chứng minh rằng lựa chọn đặc trưng là một hướng cải tiến có hiệu quả trong phạm vi thí nghiệm KNN trên WESAD này.**

EXP-017 cũng cho thấy không phải việc sử dụng càng nhiều đặc trưng càng mang lại hiệu suất tốt hơn. Việc giữ lại các đặc trưng có khả năng phân biệt cao có thể giúp KNN hoạt động hiệu quả hơn và giảm số chiều dữ liệu đầu vào.